# 📘 Chapter 03 — Embeddings: From IDs to Learned Representations
**Series: LM from First Principles**

---

## What this chapter covers

- Chapter 2 gave us vectors; this chapter turns integer token IDs into learned vectors
- Why a lookup table? The one-hot connection
- `nn.Embedding`: a learnable lookup table, nothing more
- **Core experiment:** train 2D character embeddings and observe every token move
- Cosine similarity, nearest neighbours, and movement analysis after training
- Scaling to higher dimensions: the same operations, more capacity
- Three distinct types of embedding — token, contextual, retrieval
- What positional representations add — and why token embeddings alone are not enough

In [ ]:
import torch
import torch.nn as nn                  # nn.Embedding, nn.Linear, nn.Module
import torch.nn.functional as F        # F.cross_entropy
import matplotlib.pyplot as plt        # scatter plots
import matplotlib.patches as mpatches  # legend patches for the scatter plot
import string                          # string.ascii_letters, string.punctuation
import math                            # math.log for baseline loss calculation
from collections import Counter        # character frequency counts

torch.manual_seed(42)                  # reproducible random initialisation
print('PyTorch version:', torch.__version__)

---
## From Vectors to Learned Representations

Chapter 2 gave us the mathematics of vectors: dot products, norms, cosine similarity,
and the geometry of high-dimensional spaces. But when text arrives at a language model,
it arrives as integers. The character `'A'` is token ID 65. The space character is
token ID 0. These integers carry no geometry — they are labels, not coordinates.

**How do we turn a discrete token ID into a learned vector representation?**

That is the question this chapter answers. The token IDs themselves are familiar;
from this point onward we treat them simply as indices into a learnable table.
What we are building is the mechanism that converts those indices into the kind of
real-valued vectors that Chapter 2 gave us the tools to reason about.

```
  token ID  →  [integer label]        ← this is what we have
  token ID  →  [-0.23,  0.81, ...]    ← this is what we need
```

The mechanism is an **embedding**: a learnable lookup table that maps each integer
index to a row of real-valued parameters. The vectors start random. Training moves them.
After training, the tools from Chapter 2 — dot product, cosine similarity, norm —
let us measure exactly what structure emerged and why.

---
## `nn.Embedding`: A Learnable Lookup Table

`nn.Embedding(num_embeddings, embedding_dim)` allocates a weight matrix of shape
`(num_embeddings, embedding_dim)`. When you pass it an integer tensor, it returns
the corresponding rows.

```
  weight matrix:  shape (vocab_size, C)

  embedding(torch.tensor([3, 7, 1]))
  = weight_matrix[[3, 7, 1]]
  = rows 3, 7, and 1, stacked into a (3, C) tensor

  No arithmetic during lookup. Pure indexing.
  Gradients flow back only to the rows that were accessed.
```

That last point matters for training: if a rare character never appears in a batch,
its row receives no gradient update. High-frequency characters receive more gradient
updates than rare ones. We will see this in the movement analysis below.

In [ ]:
# nn.Embedding is just a differentiable lookup table

emb_demo = nn.Embedding(6, 3)          # 6 tokens, 3 dimensions each
print('Weight matrix shape:', emb_demo.weight.shape)
print('Full weight matrix (random init):')
print(emb_demo.weight.data.round(decimals=3))
print()

# Look up three tokens by their integer IDs
tokens = torch.tensor([0, 2, 4])
result = emb_demo(tokens)              # returns rows 0, 2, 4 of the weight matrix
print('emb_demo(torch.tensor([0, 2, 4])):')
print(result.data.round(decimals=3))
print()

# Direct row indexing produces the exact same result — embedding IS indexing
print('Identical to direct row indexing:')
print(emb_demo.weight[[0, 2, 4]].data.round(decimals=3))

---
## Why a Lookup Table? The One-Hot Connection

Before `nn.Embedding`, the conceptually natural way to convert a discrete token ID
into a real-valued input was a **one-hot vector**: a vector of zeros with a single
1 at the token's position.

For a vocabulary of size V, token 2 has one-hot representation:

    [0, 0, 1, 0, 0, ...]   ← V-dimensional, 1 at index 2, 0 everywhere else

Multiplying a one-hot vector by an embedding matrix E of shape (V, C) selects exactly
one row — because every zero in the one-hot vector zeros out its corresponding row,
leaving only the row at the token's position:

    [0, 0, 1, 0, 0] @ E   =   E[2]   ← row 2 of E

`nn.Embedding` is the efficient implementation of this identity. Instead of constructing
and multiplying a large mostly-zero vector, it retrieves the row by integer index directly.
The mathematics is identical; the computation is not.

In [ ]:
# One-hot × embedding matrix = row lookup — mathematically identical

V, C = 5, 3
E    = torch.randn(V, C)          # embedding matrix, shape (V, C)

token_id = 2

# Method 1: construct the one-hot vector, then matrix multiply
one_hot    = F.one_hot(torch.tensor(token_id), num_classes=V).float()
via_matmul = one_hot @ E          # (V,) @ (V, C) → (C,)

# Method 2: direct row indexing — what nn.Embedding does internally
via_lookup = E[token_id]          # row 2 of E

print('One-hot vector:     ', one_hot.tolist())
print('Via matrix multiply:', via_matmul.round(decimals=4).tolist())
print('Via direct lookup:  ', via_lookup.round(decimals=4).tolist())
print('Results identical:  ', torch.allclose(via_matmul, via_lookup))
print()
print(f'For V={V} the win is small.')
print('For V=50,000 (a real subword vocabulary) constructing and multiplying')
print('a 50,000-dimensional mostly-zero vector is extremely wasteful.')
print('nn.Embedding skips it entirely — same result, integer index only.')

In [ ]:
# Gradients flow only to rows that were accessed during the forward pass

emb_grad_demo = nn.Embedding(6, 3)          # fresh 6-token, 3-dim table
tokens        = torch.tensor([0, 2, 4])      # look up rows 0, 2, and 4 only
result        = emb_grad_demo(tokens)        # shape: (3, 3) — only those three rows

loss = result.sum()   # any scalar derived from the output starts the backward pass
loss.backward()

print('Row  Accessed?  Will be updated?')
print('─' * 36)
for i in range(6):
    g       = emb_grad_demo.weight.grad[i]
    touched = i in [0, 2, 4]
    nonzero = g.abs().sum().item() > 1e-9
    print(f'  {i}    {str(touched):5s}      {"yes — gradient exists" if nonzero else "no  — gradient is zero"}')

print()
print('Rows 1, 3, 5 were never looked up.')
print('They receive no gradient contribution from this forward and backward pass.')

---
## From One Token to a Sequence: (B, T) → (B, T, C)

The single-token lookup above is the building block. What a language model actually
receives is a batch of sequences — multiple samples, each containing multiple tokens.
The embedding table handles this with no extra machinery: every integer in the input
tensor is independently replaced by its row from the weight matrix.

    Before embedding:

           T = 3 tokens
         ┌─────────────┐
    B=2  │  1   4   2  │    shape: (2, 3)  — one integer per position
         │  3   1   5  │
         └─────────────┘

    After embedding (C = 4):

         ┌─────────────────────────────────────────┐
    B=2  │  [v₁]  [v₄]  [v₂]                      │   shape: (2, 3, 4)
         │  [v₃]  [v₁]  [v₅]                      │   C numbers per position
         └─────────────────────────────────────────┘

Each [vᵢ] is row i from the embedding weight matrix.
Token 1 at position (0,0) and token 1 at position (1,1) return the **same row** —
the embedding is a lookup, not a function of position.

**Critical observation:** embedding does not mix tokens. Every position in the (B, T)
grid is converted independently. After this step, position 0 knows nothing about
position 1. The representation at each position is now a learned C-dimensional vector
instead of an arbitrary integer — but positions are still isolated from each other.

This is exactly the job embedding was designed to do: **representation**, not
communication. Communication between positions is the job of attention — Chapter 7.

---
## Core Experiment: Training 2D Character Embeddings

We train a minimal character-level language model with `C=2` — two embedding dimensions.
Two dimensions means every character token can be plotted as a point on a 2D plane.
We can watch every token's position before and after training.

**Model:** `Embedding(vocab_size, 2)` + `Linear(2, vocab_size)`
**Task:** predict the next character from the current character
**Data:** tinyshakespeare

```
  input char id  ->  embedding(id)  ->  2D vector
                                     ->  linear layer
                                     ->  logits over vocab
                                     ->  cross-entropy loss vs next char
```

C=2 is deliberately too small to produce a good language model.
That is not the point. The point is observability: we can see
where every token starts, and where it ends up.

> **On context length:** Each training example is one input character predicting one
> next character — T=1. The (B, T, C) shape simplifies to (B, C) here for convenience.
> The same `nn.Embedding` layer accepts full (B, T) sequences unchanged; T=1 is a
> property of this experiment, not of the embedding mechanism.

> **On vocabulary size:** Chapter 1 expanded the character set to ~95 tokens by
> unioning the corpus characters with all ASCII letters, digits, and punctuation.
> Here we use only the 65 characters that actually appear in tinyshakespeare.
> The embedding mechanism is identical; the vocabulary size is a design choice.

In [ ]:
# Load tinyshakespeare and build character vocabulary

with open('tinyshakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# Collect every unique character and sort into a stable list
chars = sorted(set(text))
vocab_size = len(chars)

# stoi: character -> integer ID  (e.g. ' ' -> 0, '!' -> 1, ...)
# itos: integer ID -> character  (reverse lookup, used when decoding output)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}

# Encode the full text as a flat tensor of integer IDs
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)

print(f'Text length:  {len(text):,} characters')
print(f'Vocab size:   {vocab_size} unique characters')
print(f'Data tensor:  {data.shape}  dtype={data.dtype}')
print()

# Count character frequencies — this matters for embedding training:
# frequent characters receive more gradient updates and converge faster
freq = Counter(text)
print('Character inventory (sorted by frequency):')
for c, count in sorted(freq.items(), key=lambda x: -x[1])[:20]:
    display = repr(c) if c in (' ', '\n', '\t') else c   # make whitespace readable
    bar = '#' * (count // 10000)                          # rough visual frequency bar
    print(f'  {display:6s}  id={stoi[c]:2d}  freq={count:7,}  {bar}')

In [ ]:
# Minimal embedding language model: lookup -> linear -> predict next character

class EmbeddingLM(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)  # ID -> vector
        self.linear    = nn.Linear(embed_dim, vocab_size)     # vector -> score per vocab token

    def forward(self, x):
        emb = self.embedding(x)   # (B,) -> (B, embed_dim)
        return self.linear(emb)   # (B, embed_dim) -> (B, vocab_size)


model_2d = EmbeddingLM(vocab_size, embed_dim=2)
print(f'Parameters: {sum(p.numel() for p in model_2d.parameters()):,}')

# Snapshot the random initial positions — compared against post-training in the scatter plot
emb_before = model_2d.embedding.weight.data.clone()

---
> **Food for thought: where does 325 come from?**
>
> In the transformer literature, tensors flowing through the model have shape **(B, T, C)**:
> B = batch size, T = sequence length (timesteps), C = embedding dimension (channels).
> C is the same 2 we passed to `EmbeddingLM`.
>
> But B and T have nothing to do with the parameter count. Parameters are the **stored
> weights** that exist in memory at all times — independent of how many samples you
> feed the model or how long the sequence is. Only `vocab_size` and `C` determine them:
>
> ```
>   nn.Embedding(65, 2)   →  65 × 2  = 130 parameters   (the lookup table)
>   nn.Linear(2, 65)      →   2 × 65 = 130 parameters   (weight matrix)
>                         →       65 =  65 parameters   (bias vector)
>                                      ─────────────────
>                                      325 parameters total
> ```
>
> During a forward pass the activation tensor takes shape **(B, T, C)**:
> the embedding outputs `C=2` numbers per token, stacked across `T` timesteps and `B`
> examples. Changing B or T changes how much work the model does per step — not how
> many weights it has.
>
> The distinction matters at scale. GPT-3 has 175 billion parameters regardless of
> whether you feed it one token or a million. Batch size and sequence length are
> *throughput* levers; parameter count is a *capacity* lever.

In [ ]:
# Training loop: learn to predict the next character from the current character

optimizer = torch.optim.Adam(model_2d.parameters(), lr=0.01)

batch_size = 256   # number of (current, next) character pairs per step
n_steps    = 5000  # total training steps
log_every  = 500   # print loss every N steps
losses     = []

for step in range(n_steps):
    # Sample a random batch of positions in the text
    ix = torch.randint(len(data) - 1, (batch_size,))
    x  = data[ix]       # current characters  — shape (batch_size,)
    y  = data[ix + 1]   # next characters     — shape (batch_size,)

    logits = model_2d(x)               # (batch_size, vocab_size) — scores over next token
    loss   = F.cross_entropy(logits, y) # scalar — average prediction error this batch
    losses.append(loss.item())

    optimizer.zero_grad()  # clear gradients from previous step
    loss.backward()        # compute new gradients
    optimizer.step()       # nudge all parameters in the direction that reduces loss

    if step % log_every == 0 or step == n_steps - 1:
        print(f'step {step:5d} | loss {loss.item():.4f}')

# Capture the embedding table as it stands after all updates
emb_after = model_2d.embedding.weight.data.clone()
print()
print(f'Training complete. Final loss: {losses[-1]:.4f}')
print(f'Random baseline (uniform):    {math.log(vocab_size):.4f}')  # loss if predicting every char equally

---
> **Food for thought: what is the best loss this model can achieve?**
>
> The training cell prints two reference points. There is a third between them —
> and understanding all three reveals something important about C.
>
> **Three levels, from worst to best:**
>
> ```
>   Uniform baseline   math.log(vocab_size) ≈ 4.17
>   ─────────────────────────────────────────────────────────────────────
>   Every character predicted equally likely. Ignores all frequency data.
>
>   Unigram entropy    H = -Σ p(c) · log p(c) ≈ 3.4–3.6
>   ─────────────────────────────────────────────────────────────────────
>   Knows character frequencies, ignores context entirely.
>   Space ' ' is very common; '$' is rare. That knowledge alone beats uniform.
>
>   True next-character entropy    theoretical floor for one-character context
>   ─────────────────────────────────────────────────────────────────────
>   Achievable by an unrestricted V×V score matrix that encodes a fully
>   independent next-character distribution per input token. Such a matrix
>   can have rank up to V.
> ```
>
> **Our C=2 model cannot reach that floor.**
>
> If predictions must factor through a C-dimensional representation, the
> token-dependent score matrix has rank at most C:
>
> ```
>   V tokens  →  C-dimensional embedding  →  V logits
>
>   Rank of E @ Wᵀ  ≤  C
> ```
>
> A shared low-dimensional linear layer creates pressure to represent similar
> predictive behaviour economically through related directions in the embedding
> space. That pressure is what produces the geometry we observe in the scatter plot.
>
> The ~2.66 final loss is what this rank-2 model achieves — above the theoretical
> floor, because the prediction table must factor through only two learned dimensions.
>
> Increasing C relaxes the constraint. At C=16 the model has more rank, more capacity
> to differentiate tokens whose next-character profiles differ subtly. That is the real
> architectural reason C=16 outperforms C=2 — not merely that we can no longer
> visualise 16 dimensions.

In [ ]:
# Scatter plot: where every token started vs where it ended up after training

def plot_embeddings(ax, embeddings, title, itos):
    for token_id, char in itos.items():
        x, y = embeddings[token_id].tolist()

        # Colour by character type so clusters are visible
        if char in string.ascii_letters:
            color, zorder = '#3a7ebf', 2   # blue  — lexical (letters)
        elif char in string.punctuation or char in (' ', '\n', '\t'):
            color, zorder = '#bf3a3a', 3   # red   — structural (punctuation, whitespace)
        else:
            color, zorder = '#888888', 1   # grey  — other

        ax.scatter(x, y, c=color, s=50, zorder=zorder, edgecolors='white', linewidth=0.3)

        # Make whitespace characters readable in the plot
        label = repr(char) if char in (' ', '\n', '\t') else char
        ax.annotate(label, (x, y), fontsize=6.5, ha='center', va='bottom',
                    xytext=(0, 3), textcoords='offset points')

    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Dimension 0')
    ax.set_ylabel('Dimension 1')
    ax.axhline(0, color='k', linewidth=0.4, alpha=0.4)
    ax.axvline(0, color='k', linewidth=0.4, alpha=0.4)
    ax.grid(True, alpha=0.2)


fig, axes = plt.subplots(1, 2, figsize=(15, 7))

plot_embeddings(axes[0], emb_before, 'Before Training (random init)', itos)
plot_embeddings(axes[1], emb_after,  'After Training (5000 steps)',   itos)

# Build legend
lex_patch   = mpatches.Patch(color='#3a7ebf', label='Lexical (letters)')
str_patch   = mpatches.Patch(color='#bf3a3a', label='Structural (punctuation, space, newline)')
other_patch = mpatches.Patch(color='#888888', label='Other')
fig.legend(handles=[lex_patch, str_patch, other_patch],
           loc='lower center', ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.02))

plt.suptitle('Character Embeddings: C=2', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('ch03_scatter.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot saved to ch03_scatter.png')

---
## Reading the Plot

**Before training (left):** All tokens are placed at random initial positions.
The position of `'a'` is unrelated to the position of `'b'` or `'.'`. No structure, no meaning.

**After training (right):** The tokens have moved. Several patterns typically emerge:

- **Lowercase letters cluster together.** The characters that tend to follow a lowercase
  letter are mostly other letters. Lowercase tokens therefore have similar successor
  distributions, and the shared linear layer benefits from placing them near each other.

- **Structural characters separate.** What tends to follow a space or a period is
  structurally different — a capital letter at a sentence start, a word-beginning letter
  after a space. That distinct successor profile pushes these tokens away from the letter cluster.

- **Uppercase letters occupy a distinct region.** After an uppercase letter the model
  almost always sees more lowercase letters (continuing a name or word). That successor
  profile differs from both lowercase mid-word and structural characters, so uppercase
  tokens land in their own region.

The model did not know any of this. It learned it entirely by trying to predict the
next character, and the embedding positions are a byproduct of that single objective.

> **What geometry means here:** Two characters end up near each other specifically when
> their *distributions over the next character* are similar — when the linear layer can
> treat them the same way without hurting prediction. This is not generic semantic
> similarity. It is the geometry of next-character prediction. Change the objective
> and you change the geometry.

> **The axes carry no intrinsic meaning.** Dimension 0 and Dimension 1 have no inherent
> interpretation. Rotate the entire embedding space and a compensating rotation of the
> linear layer produces identical predictions. What matters is the *relational structure* —
> which tokens ended up near each other. The scatter plot is a diagnostic of one learned
> parameterisation, not a unique map of linguistic structure.

> **C=2 is a lossy projection.** All the variation in character behaviour is compressed
> into two numbers per token. Some distinctions visible at C=16 or C=32 are invisible
> here. We use C=2 for observability, not performance.

In [ ]:
# Nearest neighbours by cosine similarity after training

def cosine_sim(a, b):
    # Dot product divided by product of norms — measures direction, ignores magnitude
    return (a @ b) / (a.norm() * b.norm() + 1e-8)  # 1e-8 avoids division by zero


def nearest_neighbours(embeddings, itos, target_char, n=5):
    if target_char not in stoi:
        print(f'  {repr(target_char)} not in vocab')
        return
    target_id  = stoi[target_char]
    target_vec = embeddings[target_id]

    # Score every other token against the target
    sims = []
    for i in range(len(itos)):
        if i != target_id:
            sim = cosine_sim(target_vec, embeddings[i]).item()
            sims.append((sim, i))
    sims.sort(reverse=True)  # highest cosine similarity first

    display_target = repr(target_char) if target_char in (' ', '\n', '\t') else target_char
    print(f"Nearest to '{display_target}':")
    for sim, i in sims[:n]:
        c = itos[i]
        d = repr(c) if c in (' ', '\n', '\t') else c
        print(f"  '{d}'  cosine={sim:+.4f}")
    print()


# One lexical character, one structural — enough to see the two regimes
probe_chars = ['e', '.']
print('=== Nearest neighbours AFTER training (C=2) ===\n')
for ch in probe_chars:
    nearest_neighbours(emb_after, itos, ch)

# Before training: neighbours are random — no structure expected
print('=== Nearest neighbours BEFORE training (for comparison) ===\n')
for ch in probe_chars:
    nearest_neighbours(emb_before, itos, ch)

In [ ]:
# Movement analysis: how far did each token's embedding travel during training?

movement    = (emb_after - emb_before).norm(dim=1)  # distance each row moved
norms_after = emb_after.norm(dim=1)                  # final distance from origin (for reference)

# Sort by movement — tokens that changed most appear first
sorted_ids = movement.argsort(descending=True)

print('Token movement during training (most changed → least changed):')
print(f'  Mean movement:          {movement.mean().item():.4f}')
print(f'  Mean norm after:        {norms_after.mean().item():.4f}')
print()
print(f'{"Token":>10s}  {"ID":>4s}  {"Freq%":>7s}  {"Movement":>10s}  {"Norm after":>11s}')
print('─' * 50)

for tid in sorted_ids[:15].tolist():
    c        = itos[tid]
    display  = repr(c) if c in (' ', '\n', '\t') else f"'{c}'"
    freq_pct = freq[c] / len(text) * 100
    mv       = movement[tid].item()
    na       = norms_after[tid].item()
    print(f'{display:>10s}  {tid:>4d}  {freq_pct:>7.2f}  {mv:>10.4f}  {na:>11.4f}')

print()
print('Tokens with least movement (bottom 5):')
print(f'{"Token":>10s}  {"Freq%":>7s}  {"Movement":>10s}')
print('─' * 32)
for tid in sorted_ids[-5:].tolist():
    c        = itos[tid]
    display  = repr(c) if c in (' ', '\n', '\t') else f"'{c}'"
    freq_pct = freq[c] / len(text) * 100
    mv       = movement[tid].item()
    print(f'{display:>10s}  {freq_pct:>7.2f}  {mv:>10.4f}')

---
> **Food for thought: what does `dim=1` mean?**
>
> `emb_before.norm(dim=1)` says: compute the norm **along dimension 1**, collapsing it
> to a scalar per row. The shape tells the story:
>
> ```
>   emb_before              shape: (65, 2)
>                                    ↑   ↑
>                               dim=0   dim=1
>
>   .norm(dim=1)   →   shape: (65,)
>                               ↑
>                          dim=1 collapsed; dim=0 survives
> ```
>
> For each of the 65 rows, it computes `sqrt(x₀² + x₁²)` — the length of that
> 2D vector. At C=16 it would compute `sqrt(x₀² + ... + x₁₅²)` — same idea, more
> terms. The `dim=` argument always means "reduce across this axis."
>
> This pattern appears everywhere in the transformer:
>
> ```
>   tensor.sum(dim=-1)      →  sum across the last dimension
>   tensor.mean(dim=1)      →  average across dimension 1
>   tensor.softmax(dim=-1)  →  normalise across the last dimension
>   tensor.norm(dim=1)      →  length of each row vector
> ```
>
> When you see `dim=` in later chapters, ask: which axis is being collapsed, and what
> shape survives? The answer determines what the operation means geometrically.

---
## Reading the Movement Table

Movement measures the **net displacement** of each embedding from its initialisation:
`(emb_after - emb_before).norm(dim=1)`. Frequency affects how many update opportunities
a token receives, but displacement also depends on gradient magnitudes, directions, and
optimizer dynamics — it is not a direct count of how much training occurred.

In this experiment the pattern is visible: frequent characters tend to move further.
Space, newline, and common letters appear in almost every batch and accumulate many
update opportunities over 5000 steps. Rare characters appear infrequently, receive
fewer updates within this training budget, and tend to stay closer to where they started.

The **norm after training** (final distance from origin) and **movement** often
correlate, but they measure different things:

```
  Movement   = ||emb_after − emb_before||   net displacement from initialisation
  Norm after = ||emb_after||                where the row ended up
```

A token could move from one large-norm position to another — large movement, similar
norm at both ends. Or drift steadily away from the origin — both values grow together.
Do not conflate them.

The practical consequence is structural: the embedding table trains unevenly.
Within a fixed training budget, rare tokens receive fewer update opportunities and
may be less well estimated than frequent ones. This is one motivation for the vocabulary
design choices in Chapter 4 — subword tokenisation reduces vocabulary size and smooths
frequency distributions so fewer tokens are chronically underexposed.

---
## Scaling to Higher Dimensions: C=16

C=2 is good for observation. C=16 gives each token more representational capacity —
sixteen numbers instead of two to encode its role in next-character prediction. The
model can now separate tokens whose successor profiles are subtly different, rather
than forcing all distinctions into two dimensions.

We cannot plot 16D directly, but the same operations apply:
- `nn.Embedding(vocab_size, 16)` — lookup table, shape `(vocab_size, 16)`
- `emb[i] @ emb[j]` — dot product, measures alignment
- `cosine_sim(emb[i], emb[j])` — normalised, measures direction
- `emb[i].norm()` — length of the vector in 16D space

The geometry from Chapter 2 does not change when the space gets bigger.
Only our ability to draw it does.

In [ ]:
# Train at C=16: same architecture, more capacity, richer nearest neighbours

torch.manual_seed(42)
model_16d    = EmbeddingLM(vocab_size, embed_dim=16)   # 16 dimensions per token
optimizer_16 = torch.optim.Adam(model_16d.parameters(), lr=0.01)

print('Training C=16 model...')
for step in range(5000):
    ix   = torch.randint(len(data) - 1, (256,))
    x, y = data[ix], data[ix + 1]
    loss = F.cross_entropy(model_16d(x), y)
    optimizer_16.zero_grad()
    loss.backward()
    optimizer_16.step()

    if step % 1000 == 0 or step == 4999:
        print(f'  step {step:5d} | loss {loss.item():.4f}')

# Capture the trained 16D embeddings
emb_16d = model_16d.embedding.weight.data.clone()

print()
# We cannot plot 16D, but cosine similarity and nearest neighbours work identically
print('=== Nearest neighbours at C=16 ===\n')
probe_chars_16 = ['e', 'a', 'A', 'E', '.', ',', ' ', '\n']
for ch in probe_chars_16:
    nearest_neighbours(emb_16d, itos, ch, n=4)

# Direct comparison: does higher C sharpen the similarity between punctuation?
print('--- Comparison: cosine similarity dot vs comma ---')
for label, emb in [('C=2 ', emb_after), ('C=16', emb_16d)]:
    sim = cosine_sim(emb[stoi['.']], emb[stoi[',']]).item()
    print(f"  {label}  '.' vs ',': {sim:+.4f}")

---
## Verifying the Rank Bound

The food-for-thought above made a claim: the token-dependent component of the prediction
matrix — `embedding.weight @ linear.weight.T` — has rank at most C. We can verify
this directly now that we have both trained models.

This verifies the capacity bottleneck of the token-dependent term E Wᵀ; it does not
prove a unique clustering geometry. Adding the shared bias produces the full logit
matrix L = E Wᵀ + **1**bᵀ, whose rank can be at most C+1. The C-dimensional bottleneck
therefore still strongly constrains how token identity can affect the logits.

In [ ]:
# The token-dependent logit matrix is E @ W^T — its rank is bounded by C

logit_core_2  = model_2d.embedding.weight  @ model_2d.linear.weight.T   # shape (65, 65)
logit_core_16 = model_16d.embedding.weight @ model_16d.linear.weight.T  # shape (65, 65)

rank_2  = torch.linalg.matrix_rank(logit_core_2).item()
rank_16 = torch.linalg.matrix_rank(logit_core_16).item()

print(f'Vocabulary size:          {vocab_size}')
print(f'Unrestricted matrix rank: up to {vocab_size}')
print()
print(f'C=2  model  → rank {rank_2:2d}   (ceiling: C=2)')
print(f'C=16 model  → rank {rank_16:2d}   (ceiling: C=16)')
print()
print('The token-dependent component E @ W.T has rank at most C.')
print('Full logit matrix L = E @ W.T + 1*b has rank at most C+1.')
print()
print(f'Implication: a C=2 model has at most 2 independent directions')
print(f'in its token-dependent logits, regardless of how long it trains.')

---
## Three Types of Embedding

The word "embedding" is used for several different kinds of representation in modern NLP.
It is worth separating three of them explicitly, because confusing them leads to
misapplied tools.

```
  Type 1 — Token / input embeddings
  ----------------------------------
  Maps:       token ID  ->  vector
  Shaped by:  prediction objective (next-token, masked-token)
  The vector: updated every time the token appears in a training batch.
              After training, token ID 5 always returns the same row —
              the row as it stands after all those updates.
  Examples:   our character embeddings, word2vec, GloVe, Bengio 2003
  Chapter:    THIS chapter (Ch 3)

  Type 2 — Contextual representations
  ------------------------------------
  Maps:       token in a specific context  ->  vector
  Shaped by:  full sequence via attention layers
  The vector: the same token ID begins with the same input embedding,
              but after sequence processing it can produce a different
              contextual representation depending on the surrounding
              tokens. 'bank' in 'river bank' and 'bank' in 'bank
              account' produce different contextual representations.
  Examples:   BERT hidden states, GPT hidden states
  Chapter:    Preview only here. Mechanism arrives in Ch 7 (attention).

  Type 3 — Retrieval / sentence embeddings
  -----------------------------------------
  Maps:       sentence / paragraph / document  ->  ONE vector
  Shaped by:  contrastive similarity objective
  The vector: one vector per document, not per token.
              Trained to pull similar documents together
              and push dissimilar ones apart.
  Examples:   Sentence-BERT, OpenAI text-embedding-*, Cohere Embed
  Chapter:    Touch only — named here, not implemented.
```

This chapter builds **Type 1** from scratch. Type 2 is previewed at the end.
Type 3 is named so you know where it sits — not implemented here.

---
## Positional Representations: A Preview

Token embeddings answer one question: **what is this token?**
They say nothing about **where** in the sequence the token appears.

The token embedding for `'.'` is the same vector whether it appears at position 3 or
position 300. In natural language, position carries meaning — sentence beginnings,
clause structure, word order. An embedding table has no mechanism to encode any of this.

```
  Same token, different positions:

    'bank' at position 2    →  same embedding vector
    'bank' at position 47   →  same embedding vector

  A separate mechanism must tell the model where each token sits.
```

How positional information is incorporated depends on the architecture — different
models handle it differently. We defer the full treatment to Chapter 6 (context windows)
and Chapter 7 (attention), where we have the machinery to derive it properly. The point
here is only that positional information is a separate concern from token identity, and
that token embeddings alone are not sufficient input to a sequence model.

---
## Research Connections

| Paper | What it established | Where we use it |
|---|---|---|
| Bengio et al. (2003) | Learned token embeddings as byproduct of language model training | Our entire chapter setup |
| Mikolov et al. (2013) | Geometric regularity in trained word vectors; king−man+woman≈queen | Motivation for why geometry matters |
| Pennington et al. (2014) | Global co-occurrence statistics shape geometry | Alternative training signal for Type 1 embeddings |
| Devlin et al. (2019) | Contextual representations via bidirectional attention | Preview: Type 2 mechanism arrives in Ch 7 |
| Reimers & Gurevych (2019) | Sentence-level vectors via contrastive training | Terminology context for retrieval embeddings |

---
## What This Chapter Deliberately Does Not Explain

**Skip-gram and CBOW architecture.** word2vec is named and motivated above but not
implemented. Implementing it would require a separate negative-sampling loop and
adds nothing to understanding `nn.Embedding` itself.

**Co-occurrence matrix factorisation.** GloVe's training objective is described
conceptually. The matrix factorisation maths belongs in a linear algebra course,
not here.

**Subword tokenisation.** Our vocabulary is characters. Real language models tokenise
into subwords (BPE, WordPiece, SentencePiece). This changes the vocabulary but not
the embedding mechanism. Tokenisation is Chapter 4.

**How context changes the representation.** Our embeddings are static — the same token
always maps to the same vector. Contextual representations require attention. That is
Chapter 7.

**Embedding fine-tuning vs freezing.** When you adapt a pre-trained model, you may
freeze the embedding layer or let it train. The tradeoffs depend on data size and
domain shift. This is addressed in Chapter 15 (LoRA).

**Approximate nearest-neighbour search.** FAISS, ScaNN, Annoy. The engineering
of scaling cosine similarity to millions of documents. Out of scope for this book.

**Weight tying.** Many language models share the input embedding matrix with the output
projection — the same weights that map token IDs to vectors are transposed to map hidden
states back to vocabulary scores. We keep them separate here to make the two roles
visible: `nn.Embedding` for representation, `nn.Linear` for prediction. Whether to tie
them is a model design decision, not part of understanding what an embedding is.

---
## Chapter 03 — Summary

```
  CORE INSIGHTS

  1. IDs are indices, not representations.
     A token ID is an arbitrary integer label. An embedding converts it
     to a real-valued vector whose geometry can be shaped by training.

  2. Embedding is a learnable lookup table.
     embedding(token_id) = weight_matrix[token_id]
     Equivalently: one_hot(token_id) @ weight_matrix.
     The embedding lookup itself is indexing, not a matrix multiplication.

  3. Gradients are row-local.
     Only the rows accessed in a given forward pass receive gradient
     updates. Rows never accessed in a batch are unchanged that step.

  4. (B, T) becomes (B, T, C).
     Every integer in a batch of sequences is independently replaced
     by its row from the weight matrix. Embedding changes representation,
     not communication.

  5. Training shapes representations toward the objective.
     Tokens with similar successor distributions can develop related
     geometry because the shared linear layer creates pressure to
     represent predictive similarities through the same limited dimensions.

  6. C limits representational capacity.
     The rank of E @ Wᵀ is at most C. Adding the bias raises the ceiling
     to at most C+1. The bottleneck strongly constrains how token identity
     can affect the logits.

  7. Net displacement reflects update opportunities.
     Frequent tokens tend to move farther from initialisation within a
     fixed training budget. Rare tokens may remain near their starting
     position. Displacement is not a direct count of updates.

  8. Geometry is relational; axes are arbitrary.
     Rotating the embedding space with a compensating rotation of the
     linear layer produces identical predictions. Only relational
     structure — which tokens are near which — is meaningful.

  9. Embedding solves representation, not communication.
     After this step, every position holds a richer vector than an
     integer. The positions are still isolated from each other.
     Representation: solved. Communication: still missing.

  10. Token, contextual, and retrieval embeddings are different objects.
      Same word; different training signals, different objects embedded,
      different uses.
```